# 05.2 Systematic Concurrency Experiments

Use this notebook for the formal experiment runs.

It automatically:
- rebuilds `library_sim` before every strategy,
- runs baseline, pessimistic, and optimistic tests,
- repeats each workload several times,
- saves every run to `library_results.test_run`,
- calculates averages at the end.

Keep `05.1_concurrency_tests.ipynb` as the development/debug notebook.


## 1. Imports


In [15]:
import mysql.connector
import threading
import time
import pandas as pd

from getpass import getpass


## 2. MySQL connection


In [16]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": getpass("MySQL password: ")
}

def get_connection(database=None):
    config = DB_CONFIG.copy()

    if database:
        config["database"] = database

    return mysql.connector.connect(**config)


## 3. Experiment plan

For each workload:

`total operations = threads × operations per thread`

The baseline uses the same total operations as pessimistic and optimistic.

`copy_ids = [1]` means all operations target the same copy, which creates high contention.
Later, use more copy IDs to test lower contention.


In [17]:
BATCH_NAME = "contention_comparison_v1"
REPEATS = 3

TEST_CONFIGS = [
    # LOW WORKLOAD
    {"workload": "Low", "threads": 5, "ops_per_thread": 5,
     "contention": "High", "copy_ids": [1]},

    {"workload": "Low", "threads": 5, "ops_per_thread": 5,
     "contention": "Medium", "copy_ids": [1, 2, 3, 4, 5]},

    {"workload": "Low", "threads": 5, "ops_per_thread": 5,
     "contention": "Low", "copy_ids": list(range(1, 26))},


    # MEDIUM WORKLOAD
    {"workload": "Medium", "threads": 20, "ops_per_thread": 5,
     "contention": "High", "copy_ids": [1]},

    {"workload": "Medium", "threads": 20, "ops_per_thread": 5,
     "contention": "Medium", "copy_ids": [1, 2, 3, 4, 5]},

    {"workload": "Medium", "threads": 20, "ops_per_thread": 5,
     "contention": "Low", "copy_ids": list(range(1, 26))},


    # HIGH WORKLOAD
    {"workload": "High", "threads": 50, "ops_per_thread": 20,
     "contention": "High", "copy_ids": [1]},

    {"workload": "High", "threads": 50, "ops_per_thread": 20,
     "contention": "Medium", "copy_ids": [1, 2, 3, 4, 5]},

    {"workload": "High", "threads": 50, "ops_per_thread": 20,
     "contention": "Low", "copy_ids": list(range(1, 26))},


    # STRESS WORKLOAD
    {"workload": "Stress", "threads": 50, "ops_per_thread": 100,
     "contention": "High", "copy_ids": [1]},

    {"workload": "Stress", "threads": 50, "ops_per_thread": 100,
     "contention": "Medium", "copy_ids": [1, 2, 3, 4, 5]},

    {"workload": "Stress", "threads": 50, "ops_per_thread": 100,
     "contention": "Low", "copy_ids": list(range(1, 26))}
]

for config in TEST_CONFIGS:
    total = config["threads"] * config["ops_per_thread"]

    print(
        f"{config['workload']} | "
        f"{config['contention']} contention | "
        f"{config['threads']} threads × "
        f"{config['ops_per_thread']} ops/thread = "
        f"{total:,} total operations | "
        f"{len(config['copy_ids'])} copy/copies"
    )


Low | High contention | 5 threads × 5 ops/thread = 25 total operations | 1 copy/copies
Low | Medium contention | 5 threads × 5 ops/thread = 25 total operations | 5 copy/copies
Low | Low contention | 5 threads × 5 ops/thread = 25 total operations | 25 copy/copies
Medium | High contention | 20 threads × 5 ops/thread = 100 total operations | 1 copy/copies
Medium | Medium contention | 20 threads × 5 ops/thread = 100 total operations | 5 copy/copies
Medium | Low contention | 20 threads × 5 ops/thread = 100 total operations | 25 copy/copies
High | High contention | 50 threads × 20 ops/thread = 1,000 total operations | 1 copy/copies
High | Medium contention | 50 threads × 20 ops/thread = 1,000 total operations | 5 copy/copies
High | Low contention | 50 threads × 20 ops/thread = 1,000 total operations | 25 copy/copies
Stress | High contention | 50 threads × 100 ops/thread = 5,000 total operations | 1 copy/copies
Stress | Medium contention | 50 threads × 100 ops/thread = 5,000 total operations 

## 4. Create permanent results table


In [18]:
def create_results_table():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("CREATE SCHEMA IF NOT EXISTS `library_results`")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS `library_results`.`test_run` (
            run_id INT NOT NULL AUTO_INCREMENT,
            batch_name VARCHAR(100) NOT NULL,
            workload VARCHAR(45) NOT NULL,
            strategy VARCHAR(20) NOT NULL,
            contention VARCHAR(20) NOT NULL,
            thread_count INT NOT NULL,
            operations_per_thread INT NOT NULL,
            total_operations INT NOT NULL,
            repeat_number INT NOT NULL,
            copy_count INT NOT NULL,
            runtime_seconds DECIMAL(14,6),
            successes INT,
            conflicts INT,
            deadlocks INT,
            timeouts INT,
            errors INT,
            attempt_throughput DECIMAL(14,3),
            successful_throughput DECIMAL(14,3),
            run_date DATETIME DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (run_id)
        )
    """)

    conn.commit()
    cursor.close()
    conn.close()

create_results_table()
print("Results table ready.")


Results table ready.


## 5. Rebuild `library_sim` before each strategy


In [19]:
def rebuild_library_sim():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("SET SESSION restrict_fk_on_non_standard_key = OFF")
    cursor.execute("DROP SCHEMA IF EXISTS `library_sim`")
    cursor.execute("CREATE SCHEMA `library_sim`")

    tables = [
        "Book",
        "Book_copy",
        "Member",
        "Loan",
        "Transaction_log",
        "Conflict_log",
        "Deadlock_log"
    ]

    for table in tables:
        cursor.execute(
            f"CREATE TABLE `library_sim`.`{table}` "
            f"LIKE `library`.`{table}`"
        )

        cursor.execute(
            f"INSERT INTO `library_sim`.`{table}` "
            f"SELECT * FROM `library`.`{table}`"
        )

    cursor.execute("DELETE FROM `library_sim`.`Transaction_log`")
    cursor.execute("DELETE FROM `library_sim`.`Conflict_log`")
    cursor.execute("DELETE FROM `library_sim`.`Deadlock_log`")

    conn.commit()
    cursor.close()
    conn.close()


## 6. Checkout procedures


In [20]:
BASELINE_SQL = """
CREATE PROCEDURE checkout_baseline(IN p_copy_id INT)
BEGIN
    DECLARE v_version INT;

    START TRANSACTION;

    SELECT version
    INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id;

    INSERT INTO transaction_log
    (
        operation_type,
        timestamp,
        isolation_level,
        strategy,
        success_flag,
        conflict_flag,
        deadlock_flag
    )
    VALUES
    (
        'checkout',
        NOW(),
        'READ COMMITTED',
        'baseline',
        'TRUE',
        'FALSE',
        'FALSE'
    );

    COMMIT;
END
"""

PESSIMISTIC_SQL = """
CREATE PROCEDURE checkout_pessimistic(IN p_copy_id INT)
BEGIN
    DECLARE v_version INT;

    START TRANSACTION;

    SELECT version
    INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id
    FOR UPDATE;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id;

    INSERT INTO transaction_log
    (
        operation_type,
        timestamp,
        isolation_level,
        strategy,
        success_flag,
        conflict_flag,
        deadlock_flag
    )
    VALUES
    (
        'checkout',
        NOW(),
        'READ COMMITTED',
        'pessimistic',
        'TRUE',
        'FALSE',
        'FALSE'
    );

    COMMIT;
END
"""

OPTIMISTIC_SQL = """
CREATE PROCEDURE checkout_optimistic(
    IN p_copy_id INT,
    IN p_member_id INT,
    IN p_isolation VARCHAR(45)
)
BEGIN
    DECLARE v_version INT;

    SELECT version
    INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id
      AND version = v_version;

    IF ROW_COUNT() = 0 THEN

        INSERT INTO conflict_log
        (
            old_version,
            attempted_version,
            timestamp
        )
        VALUES
        (
            v_version,
            v_version + 1,
            NOW()
        );

        INSERT INTO transaction_log
        (
            operation_type,
            timestamp,
            isolation_level,
            strategy,
            success_flag,
            conflict_flag,
            deadlock_flag
        )
        VALUES
        (
            'checkout',
            NOW(),
            p_isolation,
            'optimistic',
            'FALSE',
            'TRUE',
            'FALSE'
        );

    ELSE

        INSERT INTO transaction_log
        (
            operation_type,
            timestamp,
            isolation_level,
            strategy,
            success_flag,
            conflict_flag,
            deadlock_flag
        )
        VALUES
        (
            'checkout',
            NOW(),
            p_isolation,
            'optimistic',
            'TRUE',
            'FALSE',
            'FALSE'
        );

    END IF;

    COMMIT;
END
"""

def create_checkout_procedures():
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    procedures = [
        ("checkout_baseline", BASELINE_SQL),
        ("checkout_pessimistic", PESSIMISTIC_SQL),
        ("checkout_optimistic", OPTIMISTIC_SQL)
    ]

    for name, sql in procedures:
        cursor.execute(f"DROP PROCEDURE IF EXISTS `{name}`")
        cursor.execute(sql)

    cursor.close()
    conn.close()


## 7. Read actual successes and conflicts


In [21]:
def get_db_metrics(strategy):
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM transaction_log
        WHERE strategy = %s
          AND success_flag = 'TRUE'
    """, (strategy,))

    successes = cursor.fetchone()[0]

    cursor.execute("""
        SELECT COUNT(*)
        FROM transaction_log
        WHERE strategy = %s
          AND conflict_flag = 'TRUE'
    """, (strategy,))

    conflicts = cursor.fetchone()[0]

    cursor.close()
    conn.close()

    return successes, conflicts


## 8. Baseline runner


In [22]:
def run_baseline(total_operations, copy_ids):
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    start_time = time.perf_counter()

    for i in range(total_operations):
        copy_id = copy_ids[i % len(copy_ids)]
        cursor.callproc("checkout_baseline", [copy_id])

    runtime = time.perf_counter() - start_time

    cursor.close()
    conn.close()

    return runtime


## 9. Pessimistic runner


In [23]:
metric_lock = threading.Lock()

def run_pessimistic(num_threads, operations_per_thread, copy_ids):
    metrics = {
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    }

    def worker(thread_id):
        conn = get_connection("library_sim")
        cursor = conn.cursor()

        for operation in range(operations_per_thread):
            copy_id = copy_ids[
                (thread_id + operation) % len(copy_ids)
            ]

            try:
                cursor.callproc(
                    "checkout_pessimistic",
                    [copy_id]
                )

            except Exception as e:
                error_text = str(e)

                with metric_lock:
                    metrics["errors"] += 1

                    if "1205" in error_text:
                        metrics["timeouts"] += 1

                    if "1213" in error_text:
                        metrics["deadlocks"] += 1

                conn.rollback()

        cursor.close()
        conn.close()

    threads = []
    start_time = time.perf_counter()

    for thread_id in range(num_threads):
        thread = threading.Thread(
            target=worker,
            args=(thread_id,)
        )

        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    runtime = time.perf_counter() - start_time

    return runtime, metrics


## 10. Optimistic runner


In [24]:
def run_optimistic(num_threads, operations_per_thread, copy_ids):
    metrics = {
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    }

    def worker(thread_id):
        conn = get_connection("library_sim")
        cursor = conn.cursor()

        for operation in range(operations_per_thread):
            copy_id = copy_ids[
                (thread_id + operation) % len(copy_ids)
            ]

            try:
                cursor.callproc(
                    "checkout_optimistic",
                    [
                        copy_id,
                        thread_id,
                        "READ COMMITTED"
                    ]
                )

                conn.commit()

            except Exception as e:
                error_text = str(e)

                with metric_lock:
                    metrics["errors"] += 1

                    if "1205" in error_text:
                        metrics["timeouts"] += 1

                    if "1213" in error_text:
                        metrics["deadlocks"] += 1

                conn.rollback()

        cursor.close()
        conn.close()

    threads = []
    start_time = time.perf_counter()

    for thread_id in range(num_threads):
        thread = threading.Thread(
            target=worker,
            args=(thread_id,)
        )

        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    runtime = time.perf_counter() - start_time

    return runtime, metrics


## 11. Save each run permanently


In [25]:
def save_result(
    config,
    strategy,
    repeat_number,
    runtime,
    successes,
    conflicts,
    metrics
):
    total_operations = (
        config["threads"]
        * config["ops_per_thread"]
    )

    attempt_throughput = (
        total_operations / runtime
        if runtime > 0
        else 0
    )

    successful_throughput = (
        successes / runtime
        if runtime > 0
        else 0
    )

    conn = get_connection("library_results")
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO test_run
        (
            batch_name,
            workload,
            strategy,
            contention,
            thread_count,
            operations_per_thread,
            total_operations,
            repeat_number,
            copy_count,
            runtime_seconds,
            successes,
            conflicts,
            deadlocks,
            timeouts,
            errors,
            attempt_throughput,
            successful_throughput
        )
        VALUES
        (
            %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s
        )
    """, (
        BATCH_NAME,
        config["workload"],
        strategy,
        config["contention"],
        config["threads"],
        config["ops_per_thread"],
        total_operations,
        repeat_number,
        len(config["copy_ids"]),
        runtime,
        successes,
        conflicts,
        metrics["deadlocks"],
        metrics["timeouts"],
        metrics["errors"],
        attempt_throughput,
        successful_throughput
    ))

    conn.commit()
    cursor.close()
    conn.close()


## 12. Run the full experiment matrix

This is the main experiment cell.

For every workload and repeat:
1. fresh clone → baseline → save,
2. fresh clone → pessimistic → save,
3. fresh clone → optimistic → save.


In [26]:
for config in TEST_CONFIGS:
    total_operations = (
        config["threads"]
        * config["ops_per_thread"]
    )

    print("\n" + "=" * 80)
    print(
        f"{config['workload']} | "
        f"{config['contention']} contention | "
        f"{config['threads']} threads | "
        f"{config['ops_per_thread']} ops/thread | "
        f"{total_operations:,} total operations | "
        f"{len(config['copy_ids'])} copy/copies"
    )
    print("=" * 80)

    for repeat in range(1, REPEATS + 1):
        print(f"\nRepeat {repeat} of {REPEATS}")

        # BASELINE
        print("  Baseline...")
        rebuild_library_sim()
        create_checkout_procedures()

        baseline_time = run_baseline(
            total_operations,
            config["copy_ids"]
        )

        baseline_successes, baseline_conflicts = (
            get_db_metrics("baseline")
        )

        save_result(
            config,
            "baseline",
            repeat,
            baseline_time,
            baseline_successes,
            baseline_conflicts,
            {
                "deadlocks": 0,
                "timeouts": 0,
                "errors": 0
            }
        )

        # PESSIMISTIC
        print("  Pessimistic...")
        rebuild_library_sim()
        create_checkout_procedures()

        pess_time, pess_metrics = run_pessimistic(
            config["threads"],
            config["ops_per_thread"],
            config["copy_ids"]
        )

        pess_successes, pess_conflicts = (
            get_db_metrics("pessimistic")
        )

        save_result(
            config,
            "pessimistic",
            repeat,
            pess_time,
            pess_successes,
            pess_conflicts,
            pess_metrics
        )

        # OPTIMISTIC
        print("  Optimistic...")
        rebuild_library_sim()
        create_checkout_procedures()

        opt_time, opt_metrics = run_optimistic(
            config["threads"],
            config["ops_per_thread"],
            config["copy_ids"]
        )

        opt_successes, opt_conflicts = (
            get_db_metrics("optimistic")
        )

        save_result(
            config,
            "optimistic",
            repeat,
            opt_time,
            opt_successes,
            opt_conflicts,
            opt_metrics
        )

print("\nAll systematic experiment runs finished.")



Low | High contention | 5 threads | 5 ops/thread | 25 total operations | 1 copy/copies

Repeat 1 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 2 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 3 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Low | Medium contention | 5 threads | 5 ops/thread | 25 total operations | 5 copy/copies

Repeat 1 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 2 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 3 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Low | Low contention | 5 threads | 5 ops/thread | 25 total operations | 25 copy/copies

Repeat 1 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 2 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 3 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Medium | High contention | 20 threads | 5 ops/thread | 100 total operations | 1 copy/copies

Repeat 1 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 2 of 3
  Base

## 13. View all individual runs for this batch


In [27]:
conn = get_connection("library_results")

runs_df = pd.read_sql(
    """
    SELECT *
    FROM test_run
    WHERE batch_name = %s
    ORDER BY
        total_operations,
        contention,
        repeat_number,
        strategy
    """,
    conn,
    params=(BATCH_NAME,)
)

conn.close()

runs_df


/var/folders/qy/yl4hyv4s5kz52wg1qlh36y_w0000gn/T/ipykernel_855/527018973.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  runs_df = pd.read_sql(


,run_id,batch_name,workload,strategy,contention,thread_count,operations_per_thread,total_operations,repeat_number,copy_count,runtime_seconds,successes,conflicts,deadlocks,timeouts,errors,attempt_throughput,successful_throughput,run_date
0,109,contention_comparison_v1,Low,baseline,High,5,5,25,1,1,0.007713,25,0,0,0,0,3241.088,3241.088,2026-11-03 18:09:13
1,1,contention_comparison_v1,Low,baseline,High,5,5,25,1,1,0.006590,25,0,0,0,0,3793.435,3793.435,2026-11-03 17:45:56
2,111,contention_comparison_v1,Low,optimistic,High,5,5,25,1,1,0.007956,10,15,0,0,0,3142.233,1256.893,2026-11-03 18:09:13
3,3,contention_comparison_v1,Low,optimistic,High,5,5,25,1,1,0.009482,10,15,0,0,0,2636.598,1054.639,2026-11-03 17:45:57
4,110,contention_comparison_v1,Low,pessimistic,High,5,5,25,1,1,0.006857,25,0,0,0,0,3645.976,3645.976,2026-11-03 18:09:13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,97,contention_comparison_v1,Stress,baseline,Medium,50,100,5000,3,5,0.860747,5000,0,0,0,0,5808.906,5808.906,2026-11-03 17:46:24
212,207,contention_comparison_v1,Stress,optimistic,Medium,50,100,5000,3,5,0.727217,3968,1032,0,0,0,6875.527,5456.418,2026-11-03 18:09:44
213,99,contention_comparison_v1,Stress,optimistic,Medium,50,100,5000,3,5,0.613058,4234,766,0,0,0,8155.838,6906.364,2026-11-03 17:46:26
214,206,contention_comparison_v1,Stress,pessimistic,Medium,50,100,5000,3,5,0.749946,5000,0,0,0,0,6667.146,6667.146,2026-11-03 18:09:43


## 14. Average the repeated runs


In [28]:
summary_df = (
    runs_df
    .groupby(
        [
            "workload",
            "contention",
            "strategy",
            "thread_count",
            "operations_per_thread",
            "total_operations",
            "copy_count"
        ],
        as_index=False
    )
    .agg(
        avg_runtime_seconds=("runtime_seconds", "mean"),
        avg_successes=("successes", "mean"),
        avg_conflicts=("conflicts", "mean"),
        avg_deadlocks=("deadlocks", "mean"),
        avg_timeouts=("timeouts", "mean"),
        avg_errors=("errors", "mean"),
        avg_attempt_throughput=("attempt_throughput", "mean"),
        avg_successful_throughput=("successful_throughput", "mean")
    )
)

summary_df


,workload,contention,strategy,thread_count,operations_per_thread,total_operations,copy_count,avg_runtime_seconds,avg_successes,avg_conflicts,avg_deadlocks,avg_timeouts,avg_errors,avg_attempt_throughput,avg_successful_throughput
0,High,High,baseline,50,20,1000,1,0.165254,1000.000000,0.000000,0.0,0.0,0.0,6060.435833,6060.435833
1,High,High,optimistic,50,20,1000,1,0.127217,57.000000,943.000000,0.0,0.0,0.0,8028.795333,426.180833
2,High,High,pessimistic,50,20,1000,1,0.116738,1000.000000,0.000000,0.0,0.0,0.0,8689.491333,8689.491333
3,High,Low,baseline,50,20,1000,25,0.175394,1000.000000,0.000000,0.0,0.0,0.0,5718.985500,5718.985500
4,High,Low,optimistic,50,20,1000,25,0.171650,920.833333,79.166667,0.0,0.0,0.0,6101.937833,5647.854500
5,High,Low,pessimistic,50,20,1000,25,0.146754,1000.000000,0.000000,0.0,0.0,0.0,6859.629000,6859.629000
6,High,Medium,baseline,50,20,1000,5,0.172797,1000.000000,0.000000,0.0,0.0,0.0,5814.032333,5814.032333
7,High,Medium,optimistic,50,20,1000,5,0.209550,692.666667,307.333333,0.0,0.0,0.0,5284.250333,3704.046667
8,High,Medium,pessimistic,50,20,1000,5,0.157006,1000.000000,0.000000,0.0,0.0,0.0,6744.051000,6744.051000
9,Low,High,baseline,5,5,25,1,0.007482,25.000000,0.000000,0.0,0.0,0.0,3360.551500,3360.551500


## 15. What this notebook now tests

This version automatically tests three contention levels:

- **High contention:** 1 shared copy
- **Medium contention:** 5 shared copies
- **Low contention:** 25 shared copies

Each workload is repeated 3 times for baseline, pessimistic, and optimistic strategies.

You can still change:
- `BATCH_NAME`
- `REPEATS`
- workload sizes
- number of shared copies

All runs remain stored in `library_results.test_run`.
